# check tta layers diff

## import

In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import argparse
import random

import numpy as np
import torch
import torch.backends.cudnn as cudnn

import lavis.tasks as tasks
from lavis.common.config import Config
from lavis.common.dist_utils import get_rank, init_distributed_mode
from lavis.common.logger import setup_logger
from lavis.common.optims import (
    LinearWarmupCosineLRScheduler,
    LinearWarmupStepLRScheduler,
)
from lavis.common.utils import now

# imports modules for registration
from lavis.datasets.builders import *
from lavis.models import *
from lavis.processors import *
from lavis.runners.runner_base import RunnerBase
from lavis.tasks import *


def setup_seeds(config):
    seed = config.run_cfg.seed + get_rank()

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    cudnn.benchmark = False
    cudnn.deterministic = True

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## model_origin

In [2]:
args = argparse.Namespace(
    is_tta=True,
    cfg_path="lavis/projects/blip2/eval/ret_coco_eval_tent_debug_experiments.yaml",
    options=[]
)

job_id = now()

cfg = Config(args)

init_distributed_mode(cfg.run_cfg)

setup_seeds(cfg)

# set after init_distributed_mode() to only log on master.
setup_logger()

cfg.pretty_print()


2025-05-30 17:30:08,890 [INFO] 
=====  Running Parameters    =====
2025-05-30 17:30:08,891 [INFO] {
    "batch_size_eval": 16,
    "batch_size_train": 16,
    "device": "cuda",
    "dist_url": "env://",
    "distributed": false,
    "evaluate": true,
    "k_test": 128,
    "num_workers": 4,
    "output_dir": "output/BLIP2/Retrieval_COCO_tent_debug_experiments",
    "seed": 42,
    "task": "retrieval",
    "test_splits": [
        "test"
    ],
    "train_splits": [
        "train"
    ],
    "use_dist_eval_sampler": false,
    "valid_splits": [
        "val"
    ],
    "world_size": 1
}
2025-05-30 17:30:08,891 [INFO] 
======  Dataset Attributes  ======
2025-05-30 17:30:08,891 [INFO] 
======== coco_retrieval =======
2025-05-30 17:30:08,892 [INFO] {
    "build_info": {
        "annotations": {
            "test": {
                "md5": "3ff34b0ef2db02d01c37399f6a2a6cd1",
                "storage": "coco/annotations/coco_karpathy_test.json",
                "url": "https://storage.googl

Not using distributed mode


In [3]:
task = tasks.setup_task(cfg)
datasets = task.build_datasets(cfg)
model = task.build_model(cfg)

# runner = RunnerBase(
#     cfg=cfg, job_id=job_id, task=task, model=model, datasets=datasets
# )
# runner.evaluate(skip_reload=True)
# runner.evaluate_tta(skip_reload=True, tta_cfg=cfg.config.tta)


2025-05-30 17:30:08,900 [INFO] Building datasets...


Using downloaded and verified file: /home/zhh/ssd/excute/deeplearning/projects/datasets/coco/annotations/coco_karpathy_train.json
Using downloaded and verified file: /home/zhh/ssd/excute/deeplearning/projects/datasets/coco/annotations/coco_karpathy_val.json
Using downloaded and verified file: /home/zhh/ssd/excute/deeplearning/projects/datasets/coco/annotations/coco_karpathy_test.json


2025-05-30 17:30:09,572 [WARNING] /home/zhh/miniconda3/envs/blip2/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

2025-05-30 17:30:21,427 [WARNING] /home/zhh/ssd/excute/deeplearning/projects/TTA_MM_Retrieval/LAVIS/lavis/models/eva_vit.py:446: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer

Position interpolate from 16x16 to 26x26


2025-05-30 17:30:47,119 [WARNING] /home/zhh/ssd/excute/deeplearning/projects/TTA_MM_Retrieval/LAVIS/lavis/models/base_model.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature

## modal_adapt

In [4]:
model_adapt = task.build_model(cfg)

state_dict = torch.load("debug/tta_model_model_state_dict_after_i2tadapt_debug.pth", map_location="cpu")
model_adapt.load_state_dict(state_dict, strict=True)

Position interpolate from 16x16 to 26x26


2025-05-30 17:31:24,808 [INFO] Missing keys []
2025-05-30 17:31:24,811 [INFO] load checkpoint from /home/zhh/ssd/excute/deeplearning/projects/checkpoints/blip2_finetune_coco.pth
2025-05-30 17:31:24,915 [WARNING] /tmp/ipykernel_27659/826773678.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't ha

<All keys matched successfully>

In [5]:
model_adapt_t2i = task.build_model(cfg)

state_dict_t2i = torch.load("debug/tta_model_model_state_dict_after_t2iadapt_debug.pth", map_location="cpu")
model_adapt_t2i.load_state_dict(state_dict_t2i, strict=True)

2025-05-30 17:31:29,174 [WARNING] /home/zhh/miniconda3/envs/blip2/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

2025-05-30 17:31:40,897 [WARNING] /home/zhh/ssd/excute/deeplearning/projects/TTA_MM_Retrieval/LAVIS/lavis/models/eva_vit.py:446: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer

Position interpolate from 16x16 to 26x26


2025-05-30 17:32:06,025 [WARNING] /home/zhh/ssd/excute/deeplearning/projects/TTA_MM_Retrieval/LAVIS/lavis/models/base_model.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature

<All keys matched successfully>

## show diff layers

In [6]:
i = 0
for k,v in model.named_parameters():
    print(k, v.shape)
    # print(v)
    print(type(v))
    i+=1

    if i==5:
        break

query_tokens torch.Size([1, 32, 768])
<class 'torch.nn.parameter.Parameter'>
temp torch.Size([])
<class 'torch.nn.parameter.Parameter'>
visual_encoder.cls_token torch.Size([1, 1, 1408])
<class 'torch.nn.parameter.Parameter'>
visual_encoder.pos_embed torch.Size([1, 677, 1408])
<class 'torch.nn.parameter.Parameter'>
visual_encoder.patch_embed.proj.weight torch.Size([1408, 3, 14, 14])
<class 'torch.nn.parameter.Parameter'>


In [7]:
j = 0
# for k,v in state_dict.items():
for k,v in model_adapt.named_parameters():
    print(k, v.shape)
    # print(v)
    print(type(v))
    j+=1

    if j==5:
        break

query_tokens torch.Size([1, 32, 768])
<class 'torch.nn.parameter.Parameter'>
temp torch.Size([])
<class 'torch.nn.parameter.Parameter'>
visual_encoder.cls_token torch.Size([1, 1, 1408])
<class 'torch.nn.parameter.Parameter'>
visual_encoder.pos_embed torch.Size([1, 677, 1408])
<class 'torch.nn.parameter.Parameter'>
visual_encoder.patch_embed.proj.weight torch.Size([1408, 3, 14, 14])
<class 'torch.nn.parameter.Parameter'>


In [8]:
j = 0
name_list = []
for ((k,v),(k_adapt,v_adapt)) in zip(model.named_parameters(), model_adapt.named_parameters()):

    # print(k, v.shape)
    # print(k_adapt, v_adapt.shape)
    # print(v)
    # print(type(v))
    diff = (v - v_adapt).detach().numpy().sum()
    if diff != 0:
        print()
        print(k, k_adapt)
        print("DIFF of original & adapted : ")
        print(diff)
        name_list.append(k)

    # try:
    #     diff = (v - v_adapt).detach().numpy().sum()
    #     if diff != 0:
    #         print("DIFF of original & adapted")
    #         print(diff)
    # except:
    #     print("except while comparing original & adapted")
    #     print(v - v_adapt)

    j+=1
    # if j==5:
    #     break


Qformer.bert.embeddings.LayerNorm.weight Qformer.bert.embeddings.LayerNorm.weight
DIFF of original & adapted : 
-0.018151835

Qformer.bert.embeddings.LayerNorm.bias Qformer.bert.embeddings.LayerNorm.bias
DIFF of original & adapted : 
0.0187537

Qformer.bert.encoder.layer.0.attention.output.LayerNorm.weight Qformer.bert.encoder.layer.0.attention.output.LayerNorm.weight
DIFF of original & adapted : 
-0.07646209

Qformer.bert.encoder.layer.0.attention.output.LayerNorm.bias Qformer.bert.encoder.layer.0.attention.output.LayerNorm.bias
DIFF of original & adapted : 
0.005798539

Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.weight Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.weight
DIFF of original & adapted : 
0.003659308

Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.bias Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.bias
DIFF of original & adapted : 
0.022769786

Qformer.bert.encoder.layer.0.output_query.LayerNorm.weight Qformer

In [9]:
[ name for name in name_list if "LayerNorm" not in name]

[]

In [10]:
len(name_list)

62

In [11]:
j = 0
name_list_t2i = []
for ((k,v),(k_adapt,v_adapt)) in zip(model.named_parameters(), model_adapt_t2i.named_parameters()):

    # print(k, v.shape)
    # print(k_adapt, v_adapt.shape)
    # print(v)
    # print(type(v))
    diff = (v - v_adapt).detach().numpy().sum()
    if diff != 0:
        # print()
        # print(k, k_adapt)
        # print("DIFF of original & adapted : ")
        # print(diff)
        name_list_t2i.append(k)

    # try:
    #     diff = (v - v_adapt).detach().numpy().sum()
    #     if diff != 0:
    #         print("DIFF of original & adapted")
    #         print(diff)
    # except:
    #     print("except while comparing original & adapted")
    #     print(v - v_adapt)

    j+=1
    # if j==5:
    #     break

print([ name for name in name_list_t2i if "LayerNorm" not in name])
print(len(name_list_t2i))
print(name_list_t2i)

[]
50
['Qformer.bert.embeddings.LayerNorm.weight', 'Qformer.bert.embeddings.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.attention.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.attention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.1.attention.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.1.attention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.1.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.1.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.2.attention.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.2.attention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.2.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.2.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.3.attention.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.3.attention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.3.output.LayerNorm.weight', '

## diff of layer_list between t2i & i2t

In [12]:
diff_elements = list(set(name_list) - set(name_list_t2i))
print(len(diff_elements))

print("\r\n layers in i2t, not in t2i: ")
for e in diff_elements:
    if e not in name_list_t2i:
        print(e)

36

 layers in i2t, not in t2i: 
Qformer.bert.encoder.layer.7.output_query.LayerNorm.weight
Qformer.bert.encoder.layer.5.output_query.LayerNorm.bias
Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.bias
Qformer.bert.encoder.layer.6.output_query.LayerNorm.bias
Qformer.bert.encoder.layer.4.output_query.LayerNorm.bias
Qformer.bert.encoder.layer.6.crossattention.output.LayerNorm.bias
Qformer.bert.encoder.layer.0.output_query.LayerNorm.bias
Qformer.bert.encoder.layer.1.output_query.LayerNorm.bias
Qformer.bert.encoder.layer.6.output_query.LayerNorm.weight
Qformer.bert.encoder.layer.1.output_query.LayerNorm.weight
Qformer.bert.encoder.layer.10.output_query.LayerNorm.bias
Qformer.bert.encoder.layer.2.crossattention.output.LayerNorm.bias
Qformer.bert.encoder.layer.8.output_query.LayerNorm.weight
Qformer.bert.encoder.layer.0.output_query.LayerNorm.weight
Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.weight
Qformer.bert.encoder.layer.2.output_query.LayerNorm.weight
Qfor

In [13]:
diff_elements = list(set(name_list_t2i) - set(name_list))
print(len(diff_elements))

print("\r\n layers in t2i, not in i2t: ")
for e in diff_elements:
    if e not in name_list:
        print(e)



24

 layers in t2i, not in i2t: 
Qformer.bert.encoder.layer.7.output.LayerNorm.weight
Qformer.bert.encoder.layer.4.output.LayerNorm.bias
Qformer.bert.encoder.layer.10.output.LayerNorm.weight
Qformer.bert.encoder.layer.1.output.LayerNorm.weight
Qformer.bert.encoder.layer.7.output.LayerNorm.bias
Qformer.bert.encoder.layer.0.output.LayerNorm.weight
Qformer.bert.encoder.layer.8.output.LayerNorm.weight
Qformer.bert.encoder.layer.5.output.LayerNorm.bias
Qformer.bert.encoder.layer.6.output.LayerNorm.weight
Qformer.bert.encoder.layer.4.output.LayerNorm.weight
Qformer.bert.encoder.layer.3.output.LayerNorm.bias
Qformer.bert.encoder.layer.9.output.LayerNorm.weight
Qformer.bert.encoder.layer.8.output.LayerNorm.bias
Qformer.bert.encoder.layer.10.output.LayerNorm.bias
Qformer.bert.encoder.layer.2.output.LayerNorm.bias
Qformer.bert.encoder.layer.11.output.LayerNorm.weight
Qformer.bert.encoder.layer.0.output.LayerNorm.bias
Qformer.bert.encoder.layer.2.output.LayerNorm.weight
Qformer.bert.encoder.layer

In [14]:
name_list_layer_norm = []
for ((k,v),(k_adapt,v_adapt)) in zip(model.named_parameters(), model_adapt_t2i.named_parameters()):
    if "LayerNorm" in k:
        name_list_layer_norm.append(k)

print(len(name_list_layer_norm))
print(name_list_layer_norm)

88
['Qformer.bert.embeddings.LayerNorm.weight', 'Qformer.bert.embeddings.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.attention.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.attention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.output_query.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.1.attention.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.1.attention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.1.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.1.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.1.output_query.LayerNorm.weight', 'Qformer.bert.encoder.layer.1.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.2.attention.output

In [15]:
diff_elements_i2t_pretrain = list(set(name_list_layer_norm) - set(name_list))
print("all layer_norm list - i2t list :", len(diff_elements_i2t_pretrain))

print(diff_elements_i2t_pretrain)



all layer_norm list - i2t list : 26
['Qformer.bert.encoder.layer.7.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.4.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.1.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.10.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.7.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.8.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.6.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.5.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.4.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.3.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.9.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.8.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.10.output.LayerNorm.bias', 'Qformer.cls.predictions.transform.LayerNorm.bias', 'Qformer.bert.encoder.layer.2.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.11.output.LayerNorm.weight', 'Qformer.bert.encoder.laye

In [16]:
diff_elements_t2i_pretrain = list(set(name_list_layer_norm) - set(name_list_t2i))
print("all layer_norm list - t2i list :", len(diff_elements_t2i_pretrain))

print(diff_elements_t2i_pretrain)

all layer_norm list - t2i list : 38
['Qformer.bert.encoder.layer.7.output_query.LayerNorm.weight', 'Qformer.bert.encoder.layer.5.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.6.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.4.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.6.crossattention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.1.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.6.output_query.LayerNorm.weight', 'Qformer.bert.encoder.layer.1.output_query.LayerNorm.weight', 'Qformer.bert.encoder.layer.10.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.2.crossattention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.8.output_query.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.output_query.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'Qformer.bert.

In [ ]:
torch.cuda.empty_cache()

## conclusion

- state_dict来自于tent，由于仅在recall阶段进行tta，image和text的feature都由Qformer计算，但不进行cross-attention交互，且各自有不同的输出head；
- 当 i2t tta时，Qformer中与image_embed输出相关的output模块不参与adapt，而text_input需要经过init_hidden_states=None的cross-attention模块输出text_embed，故cross-attention参与了adapt；
- 当 t2i tta时，Qformer中与text_embed输出相关的output_query模块和cross-attention模块不参与adapt；


# analysis rerank performance

## load sims_matrix and score_i2t/t2i

In [59]:
import numpy as np

# Load the npy files
sims_i2t = np.load("debug/sims_matrix_debug.npy").transpose(1,0)
sims_t2i = np.load("debug/sims_matrix_debug.npy")
score_i2t = np.load("debug/score_matrix_i2t_debug.npy")
score_t2i = np.load("debug/score_matrix_t2i_debug.npy")

print("sims_i2t shape:", sims_i2t.shape)
print("sims_t2i shape:", sims_t2i.shape)
print("score_i2t shape:", score_i2t.shape)
print("score_t2i shape:", score_t2i.shape)

sims_i2t shape: (5000, 25010)
sims_t2i shape: (25010, 5000)
score_i2t shape: (5000, 25010)
score_t2i shape: (25010, 5000)


In [60]:
import numpy as np

# Calculate mean, max, min, median for sims_i2t
sims_i2t_mean = np.mean(sims_i2t)
sims_i2t_max = np.max(sims_i2t)
sims_i2t_min = np.min(sims_i2t)
sims_i2t_median = np.median(sims_i2t)

# Calculate mean, max, min, median for score_i2t
score_i2t_mean = np.mean(score_i2t)
score_i2t_max = np.max(score_i2t)
score_i2t_min = np.min(score_i2t)
score_i2t_median = np.median(score_i2t)

print(sims_i2t_mean, sims_i2t_max, sims_i2t_min, sims_i2t_median)
print(score_i2t_mean, score_i2t_max, score_i2t_min, score_i2t_median)

0.255471 0.6125096 0.13369465 0.25392264
-99.49673 6.178787 -100.0 -100.0


## i2t 计算topk分数

### recall

In [61]:
k = 128
## 按照sims without rerank排序
topk_idx_sims = np.argsort(-sims_i2t, axis=1)[:, :k]  # 按行取前 k 个索引
topk_sims = np.take_along_axis(sims_i2t, topk_idx_sims, axis=1)

topk_score = np.take_along_axis(score_i2t, topk_idx_sims, axis=1)
# 这里的topk_score是按照topk_sims的topk顺序进行排序的

# score = itm_score + topk_sim
itm_score = topk_score - topk_sims
# 这里的itm_score是按照topk_sims的topk顺序进行排序的

In [62]:
np.where(score_i2t > -100,1,0).sum() / 5000

128.0

In [63]:
topk_sims[0][:10], itm_score[0][:10], topk_score[0][:10]

(array([0.5554966 , 0.5444903 , 0.53652364, 0.5284449 , 0.5192349 ,
        0.5135941 , 0.5107927 , 0.49589097, 0.4883941 , 0.48527563],
       dtype=float32),
 array([ 3.5666833 ,  3.7053866 ,  1.9656584 ,  2.9167795 ,  0.724301  ,
         0.64915115,  0.27841288, -0.502455  , -0.1639854 , -0.06578484],
       dtype=float32),
 array([ 4.12218   ,  4.249877  ,  2.502182  ,  3.4452243 ,  1.2435359 ,
         1.1627452 ,  0.78920555, -0.00656402,  0.3244087 ,  0.41949078],
       dtype=float32))

In [64]:
itm_score.shape

(5000, 128)

In [65]:
print(np.mean(topk_sims),np.max(topk_sims),np.min(topk_sims),np.median(topk_sims))
print(np.mean(itm_score),np.max(itm_score),np.min(itm_score),np.median(itm_score))
print(np.mean(topk_score),np.max(topk_score),np.min(topk_score),np.median(topk_score))

0.4083039 0.6125096 0.29702246 0.4017635
-2.0393946 5.6112356 -100.456314 -1.7661903
-1.6310914 6.178787 -100.0 -1.3629978


In [66]:
topk_idx_sims[0][:10]

array([    4,     3,     1,     0,  5696, 13804, 13806, 23576, 11859,
           2])

### rerank

In [70]:
## 按照itm_score only rerank排序
topk_idx_itm_score = np.argsort(-(score_i2t -sims_i2t), axis=1)[:, :k]  # 按行取前 k 个索引
topk_itm_score = np.take_along_axis((score_i2t-sims_i2t), topk_idx_itm_score, axis=1)

In [71]:
topk_idx_itm_score[0][:10]

array([    3,     4,     0,     1,  5696, 13804, 13806, 15789, 23560,
       15790])

### recall + rerank

In [72]:
## 按照score recall + rerank排序
topk_idx_score = np.argsort(-score_i2t, axis=1)[:, :k]  # 按行取前 k 个索引
topk_score = np.take_along_axis(score_i2t, topk_idx_score, axis=1)


In [73]:
topk_idx_score[0,:10]

array([    3,     4,     0,     1,  5696, 13804, 13806, 15789, 23560,
       15790])

### 加载 test label 分别计算分数

In [74]:
i2t_label = datasets["coco_retrieval"]["test"].img2txt

In [75]:
import numpy as np

def calculate_recall(topk_idx_sims, i2t_label):
    total_samples = len(i2t_label)
    recall_at_1 = 0
    recall_at_5 = 0
    recall_at_10 = 0

    for i in range(total_samples):
        true_label = i2t_label[i]
        topk_indices = topk_idx_sims[i]
        r1_flag = False
        r5_flag = False
        r10_flag = False

        for label in true_label:
            if label in topk_indices[:1].tolist():
                r1_flag = True
            if label in topk_indices[:5].tolist():
                r5_flag = True
            if label in topk_indices[:10].tolist():
                r10_flag = True

        if r1_flag:
            recall_at_1 += 1
        if r5_flag:
            recall_at_5 += 1
        if r10_flag:
            recall_at_10 += 1

    recall_at_1 /= total_samples
    recall_at_5 /= total_samples
    recall_at_10 /= total_samples

    return recall_at_1, recall_at_5, recall_at_10

In [76]:
topk_idx_sims[0][:10].tolist()

[4, 3, 1, 0, 5696, 13804, 13806, 23576, 11859, 2]

In [77]:
i2t_label[0]

[0, 1, 2, 3, 4]

In [78]:
# 调用方法并输出结果
recall_at_1, recall_at_5, recall_at_10 = calculate_recall(topk_idx_sims, i2t_label)
print("recall")
print("    Recall@1: {:.4f}".format(recall_at_1), "Recall@5: {:.4f}".format(recall_at_5), "Recall@10: {:.4f}".format(recall_at_10))
print()

recall_at_1, recall_at_5, recall_at_10 = calculate_recall(topk_idx_itm_score, i2t_label)
print("rerank")
print("    Recall@1: {:.4f}".format(recall_at_1), "Recall@5: {:.4f}".format(recall_at_5), "Recall@10: {:.4f}".format(recall_at_10))
print()

recall_at_1, recall_at_5, recall_at_10 = calculate_recall(topk_idx_score, i2t_label)
print("recall+rerank")
print("    Recall@1: {:.4f}".format(recall_at_1), "Recall@5: {:.4f}".format(recall_at_5), "Recall@10: {:.4f}".format(recall_at_10))
print()

recall
    Recall@1: 0.7436 Recall@5: 0.9424 Recall@10: 0.9742

rerank
    Recall@1: 0.8552 Recall@5: 0.9694 Recall@10: 0.9844

recall+rerank
    Recall@1: 0.8542 Recall@5: 0.9702 Recall@10: 0.9848



## t2i 计算topk分数

### recall

In [80]:
k = 128
## 按照sims without rerank排序
topk_idx_sims = np.argsort(-sims_t2i, axis=1)[:, :k]  # 按行取前 k 个索引
topk_sims = np.take_along_axis(sims_t2i, topk_idx_sims, axis=1)

topk_score = np.take_along_axis(score_t2i, topk_idx_sims, axis=1)
# 这里的topk_score是按照topk_sims的topk顺序进行排序的

# score = itm_score + topk_sim
itm_score = topk_score - topk_sims
# 这里的itm_score是按照topk_sims的topk顺序进行排序的

In [97]:
np.where(score_t2i > -100,1,0).sum() / 25010

128.0

In [82]:
topk_sims[0][:10], itm_score[0][:10], topk_score[0][:10]

(array([0.5284449 , 0.40552175, 0.40464383, 0.39618874, 0.38629746,
        0.3814321 , 0.37906712, 0.37194595, 0.37108356, 0.3689754 ],
       dtype=float32),
 array([ 2.9167795, -5.800764 , -6.152039 , -5.662221 , -6.0254555,
        -5.8958883, -6.414954 , -5.880032 , -6.2413235, -5.3765373],
       dtype=float32),
 array([ 3.4452243, -5.395242 , -5.747395 , -5.266032 , -5.6391582,
        -5.5144563, -6.0358872, -5.508086 , -5.8702397, -5.0075617],
       dtype=float32))

In [83]:
itm_score.shape

(25010, 128)

In [84]:
print(np.mean(topk_sims),np.max(topk_sims),np.min(topk_sims),np.median(topk_sims))
print(np.mean(itm_score),np.max(itm_score),np.min(itm_score),np.median(itm_score))
print(np.mean(topk_score),np.max(topk_score),np.min(topk_score),np.median(topk_score))

0.3304176 0.6125096 0.22253025 0.3164438
-3.9862561 5.6112356 -100.26652 -4.2982965
-3.655837 6.178787 -100.0 -3.978891


In [85]:
topk_idx_sims[0][:10]

array([   0,  788, 2761, 4100, 2371,  513, 1485, 3971, 2762, 3156])

### rerank

In [88]:
## 按照itm_score only rerank排序
topk_idx_itm_score = np.argsort(-(score_t2i -sims_t2i), axis=1)[:, :k]  # 按行取前 k 个索引
topk_itm_score = np.take_along_axis((score_t2i-sims_t2i), topk_idx_itm_score, axis=1)

In [89]:
topk_idx_itm_score[0][:10]

array([   0, 4833,   58, 3433, 2369, 4713,   21,  708, 1212, 2887])

### recall + rerank

In [90]:
## 按照score recall + rerank排序
topk_idx_score = np.argsort(-score_t2i, axis=1)[:, :k]  # 按行取前 k 个索引
topk_score = np.take_along_axis(score_t2i, topk_idx_score, axis=1)


In [91]:
topk_idx_score[0,:10]

array([   0, 4833,   58, 3433,   21, 2369, 4713,  708, 1212, 4830])

### 加载 test label 分别计算分数

In [92]:
t2i_label = datasets["coco_retrieval"]["test"].txt2img

In [99]:
import numpy as np

def calculate_recall(topk_idx_sims, t2i_label):
    total_samples = len(t2i_label)
    recall_at_1 = 0
    recall_at_5 = 0
    recall_at_10 = 0

    for i in range(total_samples):
        true_label = t2i_label[i]
        topk_indices = topk_idx_sims[i]
        if true_label in topk_indices[:1].tolist():
            recall_at_1 += 1
        if true_label in topk_indices[:5].tolist():
            recall_at_5 += 1
        if true_label in topk_indices[:10].tolist():
            recall_at_10 += 1

    recall_at_1 /= total_samples
    recall_at_5 /= total_samples
    recall_at_10 /= total_samples

    return recall_at_1, recall_at_5, recall_at_10

In [100]:
topk_idx_sims[0][:10].tolist()

[0, 788, 2761, 4100, 2371, 513, 1485, 3971, 2762, 3156]

In [101]:
t2i_label[0]

0

In [102]:
# 调用方法并输出结果
recall_at_1, recall_at_5, recall_at_10 = calculate_recall(topk_idx_sims, t2i_label)
print("recall")
print("    Recall@1: {:.4f}".format(recall_at_1), "Recall@5: {:.4f}".format(recall_at_5), "Recall@10: {:.4f}".format(recall_at_10))
print()

recall_at_1, recall_at_5, recall_at_10 = calculate_recall(topk_idx_itm_score, t2i_label)
print("rerank")
print("    Recall@1: {:.4f}".format(recall_at_1), "Recall@5: {:.4f}".format(recall_at_5), "Recall@10: {:.4f}".format(recall_at_10))
print()

recall_at_1, recall_at_5, recall_at_10 = calculate_recall(topk_idx_score, t2i_label)
print("recall+rerank")
print("    Recall@1: {:.4f}".format(recall_at_1), "Recall@5: {:.4f}".format(recall_at_5), "Recall@10: {:.4f}".format(recall_at_10))
print()

recall
    Recall@1: 0.6351 Recall@5: 0.8608 Recall@10: 0.9185

rerank
    Recall@1: 0.6820 Recall@5: 0.8766 Recall@10: 0.9248

recall+rerank
    Recall@1: 0.6825 Recall@5: 0.8773 Recall@10: 0.9263



: 

## analysis

- 在cos_sim不除以temperature的情况下，cos_sim的取值范围为-1,1；而itm_score的取值范围为0,15以上；
- 二者量纲并不相同，而在blip和blip2中，作者的代码将两者相加，达成了使用cos_sim微调itm_score的效果；
- 根据metric在i2t中仅使用itm_score，recall@1会上升，但recall@5和10会下降；
- 根据metric在t2i中仅使用itm_score，recall@1,recall@5,recall@10均会下降；

# Label Smoothing Strategy

In [ ]:
import torch
import numpy as np
import random

random_list = [random.uniform(0, 0.01) for _ in range(123)]
print(random_list)
topk_list = [0.51, 0.5, 0.45, 0.42, 0.41].extend(random_list)
print(topk_list)

[0.009217353198419926, 0.0005728912500074035, 0.009955697807776987, 0.00411013362104183, 0.009617229749585255, 0.008493844629943055, 0.006279192691278868, 0.0011056480080931775, 0.00011042847433467728, 0.007674901915110379, 0.0065957661017722715, 0.006123737407247171, 0.0036353823998097256, 0.0029145039084063243, 0.00381795142661739, 0.006339325041557745, 0.006894175883219745, 0.007927051152253919, 0.006194864495813938, 0.001054733126479004, 0.005946354194485495, 0.00055391048989485, 0.0005280534573991658, 0.004948562089221742, 0.007941493298491557, 0.009907494157816483, 0.005473847050475552, 0.00991474723511903, 0.007104610040356662, 0.009639657004252691, 0.006215034486152662, 0.005103236878501964, 0.006832880808111512, 0.0061601923298847255, 0.0033844231197398544, 0.006063886632548293, 0.007946645100676254, 0.007008017272813296, 0.0019968927685815817, 0.009791483961218551, 0.0039296875447135, 0.006691909376281834, 0.00979838075640735, 0.006290641272214204, 0.006455357107362546, 0.006

In [ ]:
topk_list = [0.51, 0.5, 0.45, 0.42, 0.41].extend(random_list)
topk_tensor = torch.Tensor([])

In [ ]:
import random

random_list = [random.uniform(0, 0.01) for _ in range(123)]
print(random_list)